# ECG Training Notebook — DAT255

This notebook is responsible for training the CNN and BiLSTM models, tuning hyperparameters with Keras Tuner,
and saving all artifacts needed for the final evaluation and comparison in `comparison.ipynb`.
The training runs are somewhat expensive, so the notebook is structured so that you can run it once to produce the final models and training histories, and then re-run `comparison.ipynb` as many times as you like without needing to re-train
(since the `comparison.ipynb` notebook only consumes the artifacts it saves)

**Pipeline:**
1. Setup & reproducibility
2. Load dataset + build multi-hot labels
3. Train/val/test split (PTB-XL folds), normalisation, class weights
4. Data augmentation (time shift, Gaussian noise, amplitude scaling)
5. Model definitions — 1D-CNN, BiLSTM
6. Focal loss
7. Keras Tuner hyperparameter search on CNN and BiLSTM (separate searches)
8. Final training runs — each model trained twice (with and without augmentation)
9. Save everything the comparison notebook will need

**Artifacts produced** (in `artifacts/`):
- `ecg_cnn_aug_final.keras`, `ecg_cnn_noaug_final.keras`
- `ecg_lstm_aug_final.keras`, `ecg_lstm_noaug_final.keras`
- `normalisation_params.npz`
- `eval_sets.npz` (X_val, y_val, X_test, y_test — for comparison notebook)
- `histories.pkl` (training curves for all runs)
- `tuner_results.json` (CNN tuner summary)
- `tuner_results_lstm.json` (BiLSTM tuner summary)


## 1. Setup

In [1]:
# Colab setup — run once
# !pip install -q wfdb keras tensorflow mlflow keras-tuner

In [2]:
import os, json, pickle, random, zipfile, ast
os.environ.pop("MLFLOW_TRACKING_URI", None)
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import numpy as np
import pandas as pd
import wfdb
import tensorflow as tf
import keras
from keras import layers, callbacks, optimizers
import keras_tuner as kt
from sklearn.preprocessing import MultiLabelBinarizer
import warnings
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

# Artifact directory
ART_DIR = "artifacts"
os.makedirs(ART_DIR, exist_ok=True)

tf.autograph.set_verbosity(0)
warnings.filterwarnings("ignore")

print(f"Keras   : {keras.__version__}")
print(f"TF      : {tf.__version__}")
print(f"Tuner   : {kt.__version__}")
print(f"Backend : {keras.backend.backend()}")

Keras   : 3.13.2
TF      : 2.21.0
Tuner   : 1.4.8
Backend : tensorflow


## 2. Load dataset & build labels

In [3]:
# Paths — adjust for your environment
ZIP_PATH = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip"
DATA_DIR = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

if not os.path.isdir(DATA_DIR):
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(".")
    print("Done.")

# Load metadata
df = pd.read_csv(os.path.join(DATA_DIR, "ptbxl_database.csv"), index_col="ecg_id")
df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)

# SCP -> superclass mapping
scp_df = pd.read_csv(os.path.join(DATA_DIR, "scp_statements.csv"), index_col=0)
scp_df = scp_df[scp_df["diagnostic"] == 1]
code_to_superclass = scp_df["diagnostic_class"].to_dict()

SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
NUM_CLASSES = len(SUPERCLASSES)

def extract_superclasses(scp_dict):
    return list({code_to_superclass[c] for c in scp_dict
                 if c in code_to_superclass})

df["superclasses"] = df["scp_codes"].apply(extract_superclasses)
df = df[df["superclasses"].map(len) > 0].copy()   # drop unlabelled
df = df[df["age"] <= 120].copy()                  # drop age outliers

mlb = MultiLabelBinarizer(classes=SUPERCLASSES)
labels = mlb.fit_transform(df["superclasses"])
df[SUPERCLASSES] = labels

print(f"Samples after filtering: {len(df)}")
print("Class distribution:")
for i, sc in enumerate(SUPERCLASSES):
    n = labels[:, i].sum()
    print(f"  {sc:5s}: {n:5d}  ({n/len(df)*100:5.1f}%)")

Samples after filtering: 21103
Class distribution:
  NORM :  9483  ( 44.9%)
  MI   :  5348  ( 25.3%)
  STTC :  5099  ( 24.2%)
  CD   :  4756  ( 22.5%)
  HYP  :  2593  ( 12.3%)


In [4]:
# Load signals (100 Hz, 1000 samples, 12 leads)

SAMPLING_RATE = 100
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF",
              "V1", "V2", "V3", "V4", "V5", "V6"]

def _read_one(path):
    return wfdb.rdrecord(path).p_signal.astype(np.float32)

def load_raw_signals(df, data_dir, cache_path=None, n_workers=8):
    """Load PTB-XL signals with parallel I/O and optional disk cache."""
    if cache_path and os.path.exists(cache_path):
        print(f"Loading cached signals from {cache_path}")
        return np.load(cache_path)

    col = "filename_lr"  # 100 Hz version
    paths = [os.path.join(data_dir, p) for p in df[col].values]

    N = len(paths)
    out = np.empty((N, 1000, 12), dtype=np.float32)

    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        for i, sig in enumerate(tqdm(pool.map(_read_one, paths),
                                     total=N, desc="Reading WFDB")):
            out[i] = sig

    if cache_path:
        print(f"Caching to {cache_path}")
        np.save(cache_path, out)

    return out

print("Loading ECG signals...")
cache_path = os.path.join(ART_DIR, "ptbxl_signals_100hz.npy")
X_all = load_raw_signals(df, DATA_DIR, cache_path=cache_path)
y_all = labels.astype(np.float32)
print(f"X_all: {X_all.shape}   y_all: {y_all.shape}")

Loading ECG signals...


Reading WFDB:   0%|          | 0/21103 [00:00<?, ?it/s]

Caching to artifacts/ptbxl_signals_100hz.npy
X_all: (21103, 1000, 12)   y_all: (21103, 5)


## 3. Split, normalise, class weights

In [5]:
# PTB-XL official split: folds 1-8 train, 9 val, 10 test
folds = df["strat_fold"].values
train_mask = folds <= 8
val_mask   = folds == 9
test_mask  = folds == 10

X_train_raw, y_train = X_all[train_mask], y_all[train_mask]
X_val_raw,   y_val   = X_all[val_mask],   y_all[val_mask]
X_test_raw,  y_test  = X_all[test_mask],  y_all[test_mask]

# Per-channel z-score (fit on train only)
train_mean = X_train_raw.mean(axis=(0, 1), keepdims=True)
train_std  = X_train_raw.std(axis=(0, 1), keepdims=True)
train_std[train_std == 0] = 1.0

def normalise(X):
    return np.clip((X - train_mean) / train_std, -10.0, 10.0)

X_train = normalise(X_train_raw)
X_val   = normalise(X_val_raw)
X_test  = normalise(X_test_raw)

# Save normalisation + test set for the comparison notebook
np.savez(os.path.join(ART_DIR, "normalisation_params.npz"),
         mean=train_mean, std=train_std)
np.savez(os.path.join(ART_DIR, "eval_sets.npz"),
         X_val=X_val, y_val=y_val,
         X_test=X_test, y_test=y_test)

print(f"Train: {X_train.shape}   Val: {X_val.shape}   Test: {X_test.shape}")

Train: (16872, 1000, 12)   Val: (2105, 1000, 12)   Test: (2126, 1000, 12)


In [6]:
# Class weights for multi-label BCE (neg/pos ratio per class)
pos = y_train.sum(axis=0)
neg = len(y_train) - pos
class_weights = neg / (pos + 1e-7)

print("Class weights (neg/pos ratio):")
for sc, w in zip(SUPERCLASSES, class_weights):
    print(f"  {sc:5s}: {w:.2f}")

Class weights (neg/pos ratio):
  NORM : 1.23
  MI   : 2.93
  STTC : 3.13
  CD   : 3.43
  HYP  : 7.12


## 4. Data augmentation

Three cheap, signal-preserving augmentations applied on the fly during training:
- **Random time shift** — rolls the signal by ±50 samples (±0.5s)
- **Gaussian noise** — σ=0.02 on normalised signal
- **Amplitude scaling** — random factor in [0.9, 1.1]

These simulate real-world variation (slightly different recording start times,
sensor noise, patient-to-patient amplitude differences) without changing the
diagnostic content. All are applied with 50% probability per-sample.

In [7]:
def augment_ecg(x, y):
    """tf.data augmentation — x shape (T, 12), y shape (5,)."""
    # Random time shift (±50 samples)
    if tf.random.uniform(()) < 0.5:
        shift = tf.random.uniform((), -50, 51, dtype=tf.int32)
        x = tf.roll(x, shift, axis=0)

    # Gaussian noise
    if tf.random.uniform(()) < 0.5:
        x = x + tf.random.normal(tf.shape(x), mean=0.0, stddev=0.02)

    # Amplitude scaling
    if tf.random.uniform(()) < 0.5:
        scale = tf.random.uniform((), 0.9, 1.1)
        x = x * scale

    return x, y


def make_dataset(X, y, batch_size=64, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(X), 2048), seed=SEED)
    if augment:
        ds = ds.map(augment_ecg, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


BATCH_SIZE = 64
train_ds = make_dataset(X_train, y_train, BATCH_SIZE, augment=True, shuffle=True)
val_ds   = make_dataset(X_val, y_val, BATCH_SIZE)
test_ds  = make_dataset(X_test, y_test, BATCH_SIZE)

## 5. Model definitions

Two architectures, as specified in the project pre-approval:
- **1D-CNN** — 4 conv blocks (wide-to-narrow kernels), GAP, dense head
- **BiLSTM** — conv front-end + 2× bidirectional LSTM

Each will be trained twice: once with augmentation and once without, to ablate
whether augmentation actually helps on this dataset (section 8).


In [8]:
INPUT_SHAPE = (1000, 12)


def _last_conv_name(model):
    """Return the name of the last Conv1D layer — use this as the Grad-CAM target."""
    return next(l.name for l in reversed(model.layers)
                if isinstance(l, layers.Conv1D))


def conv_block(x, filters, kernel_size, spatial_dropout=0.0, pool=True):
    """Conv -> BN -> ReLU -> (MaxPool) -> (SpatialDropout).

    use_bias=False because BN has its own shift parameter.
    SpatialDropout1D drops whole channels, which preserves spatial structure
    in feature maps (plain Dropout breaks it).
    """
    x = layers.Conv1D(filters, kernel_size, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    if pool:
        x = layers.MaxPooling1D(pool_size=2)(x)
    if spatial_dropout > 0:
        x = layers.SpatialDropout1D(spatial_dropout)(x)
    return x


def build_cnn(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
              filters=(32, 64, 128, 256), dropout=0.3, dense_units=128):
    """Baseline 1D-CNN for PTB-XL (100 Hz, 12 leads).

    ECG-specific choices:
      * Wide first kernel (15 samples = 150 ms) to span a full QRS complex.
      * SpatialDropout1D only in deeper conv blocks; plain Dropout on Dense.
      * No magic layer names — use _last_conv_name(model) for Grad-CAM.
    """
    inputs = keras.Input(shape=input_shape, name="ecg_input")
    x = inputs
    kernels = [15, 7, 5, 3]  # wide -> narrow
    for i, f in enumerate(filters):
        k = kernels[i] if i < len(kernels) else 3
        sd = 0.1 if i >= 2 else 0.0  # only the deeper blocks get spatial dropout
        x = conv_block(x, f, k, spatial_dropout=sd)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="sigmoid", name="predictions")(x)
    return keras.Model(inputs, outputs, name="cnn_1d")


def build_lstm(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES,
               dropout=0.3, dense_units=128,
               lstm_units_1=128, lstm_units_2=64):
    """Conv feature extractor -> BiLSTM.

    Three conv stages reduce 1000 timesteps to ~62 before the LSTM, so the
    recurrent layers operate on rich features instead of quasi-raw signal.

    Tunable hyperparameters: dropout, dense_units, lstm_units_1, lstm_units_2.
    """
    inputs = keras.Input(shape=input_shape, name="ecg_input")
    x = conv_block(inputs, 64, 15, spatial_dropout=0.0)   # 1000 -> 500
    x = conv_block(x,     128,  7, spatial_dropout=0.0)   #  500 -> 250
    x = conv_block(x,     128,  5, spatial_dropout=0.1)   #  250 -> 125
    x = layers.MaxPooling1D(2)(x)                         #  125 ->  62

    x = layers.Bidirectional(
        layers.LSTM(lstm_units_1, return_sequences=True, dropout=dropout))(x)
    x = layers.Bidirectional(
        layers.LSTM(lstm_units_2, dropout=dropout))(x)

    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="sigmoid", name="predictions")(x)
    return keras.Model(inputs, outputs, name="bilstm")


# Quick summary
for build_fn in [build_cnn, build_lstm]:
    m = build_fn()
    print(f"{m.name:12s}  params={m.count_params():>10,}  "
          f"gradcam_layer={_last_conv_name(m)}")
    del m


cnn_1d        params=   194,821  gradcam_layer=conv1d_3
bilstm        params=   596,741  gradcam_layer=conv1d_6


## 6. Focal Loss

Binary cross-entropy treats every sample equally. Focal loss down-weights
easy examples (where the model is already confident) and focuses gradient
on hard examples — particularly useful for rare classes like HYP (~2% of PTB-XL).

$\text{FL}(p_t) = -\alpha (1-p_t)^\gamma \log(p_t)$

We use α=0.25, γ=2.0 (the Lin et al. defaults from the original paper).

In [9]:
def binary_focal_loss(alpha=0.25, gamma=2.0):
    """Multi-label focal loss with sigmoid outputs."""
    def loss_fn(y_true, y_pred):
        eps = keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1.0 - y_pred)
        alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1.0 - alpha)
        return -tf.reduce_mean(alpha_t * tf.pow(1.0 - pt, gamma) * tf.math.log(pt))
    loss_fn.__name__ = "binary_focal_loss"
    return loss_fn

## 7. Keras Tuner — hyperparameter search on CNN and BiLSTM

Both architectures are tuned (separate searches to keep configs independent).
**Hyperband** is used because it kills bad trials early — much faster than
random search. `max_epochs=20` per trial, `factor=3`.

### CNN search space

| Hyperparam      | Range                                        |
|-----------------|----------------------------------------------|
| filters\_mult   | [0.5, 1.0, 1.5, 2.0]   → scales (32, 64, 128, 256) |
| dropout         | [0.2, 0.3, 0.4, 0.5]                         |
| dense\_units    | [64, 128, 256]                              |
| learning\_rate  | log-uniform [1e-4, 3e-3]                    |
| loss            | {BCE, focal}                                 |

### BiLSTM search space

| Hyperparam      | Range                                        |
|-----------------|----------------------------------------------|
| lstm\_units\_1  | [64, 128, 256]                              |
| lstm\_units\_2  | [32, 64, 128]                               |
| dropout         | [0.2, 0.3, 0.4, 0.5]                         |
| dense\_units    | [64, 128, 256]                              |
| learning\_rate  | log-uniform [1e-4, 3e-3]                    |
| loss            | {BCE, focal}                                 |


In [10]:
TUNE = True        # Set to False to skip tuning and use defaults
TUNER_MAX_EPOCHS = 20
TUNER_DIR = "keras_tuner"

# --- CNN tuner --------------------------------------------------------
def build_tunable_cnn(hp):
    mult = hp.Choice("filters_mult", [0.5, 1.0, 1.5, 2.0])
    dropout = hp.Float("dropout", 0.2, 0.5, step=0.1)
    dense_units = hp.Choice("dense_units", [64, 128, 256])
    lr = hp.Float("lr", 1e-4, 3e-3, sampling="log")
    loss_type = hp.Choice("loss", ["bce", "focal"])

    base_filters = (32, 64, 128, 256)
    filters = tuple(max(8, int(f * mult)) for f in base_filters)

    model = build_cnn(filters=filters, dropout=dropout, dense_units=dense_units)
    loss = binary_focal_loss() if loss_type == "focal" else "binary_crossentropy"
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=loss,
        metrics=[keras.metrics.AUC(name="auc", multi_label=True)],
    )
    return model


if TUNE:
    tuner_cnn = kt.Hyperband(
        build_tunable_cnn,
        objective=kt.Objective("val_auc", direction="max"),
        max_epochs=TUNER_MAX_EPOCHS,
        factor=3,
        directory=TUNER_DIR,
        project_name="ecg_cnn",
        overwrite=False,
        seed=SEED,
    )
    tuner_cnn.search_space_summary()


Search space summary
Default search space size: 5
filters_mult (Choice)
{'default': 0.5, 'conditions': [], 'values': [0.5, 1.0, 1.5, 2.0], 'ordered': True}
dropout (Float)
{'default': 0.2, 'conditions': [], 'min_value': 0.2, 'max_value': 0.5, 'step': 0.1, 'sampling': 'linear'}
dense_units (Choice)
{'default': 64, 'conditions': [], 'values': [64, 128, 256], 'ordered': True}
lr (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.003, 'step': None, 'sampling': 'log'}
loss (Choice)
{'default': 'bce', 'conditions': [], 'values': ['bce', 'focal'], 'ordered': False}


In [11]:
if TUNE:
    tuner_cnn.search(
        train_ds,
        validation_data=val_ds,
        epochs=TUNER_MAX_EPOCHS,
        callbacks=[callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                           patience=4, restore_best_weights=True)],
        verbose=1,
    )
    best_hps_cnn = tuner_cnn.get_best_hyperparameters(num_trials=1)[0]
    best_config_cnn = {k: best_hps_cnn.get(k) for k in
                       ["filters_mult", "dropout", "dense_units", "lr", "loss"]}
    print("Best CNN hyperparameters:", best_config_cnn)

    with open(os.path.join(ART_DIR, "tuner_results.json"), "w") as f:
        json.dump(best_config_cnn, f, indent=2)
else:
    best_config_cnn = {"filters_mult": 1.0, "dropout": 0.3,
                       "dense_units": 128, "lr": 1e-3, "loss": "focal"}
    print("CNN tuning skipped. Using defaults:", best_config_cnn)


Trial 30 Complete [00h 00m 53s]
val_auc: 0.9232400059700012

Best val_auc So Far: 0.9272116422653198
Total elapsed time: 00h 12m 27s
Best CNN hyperparameters: {'filters_mult': 2.0, 'dropout': 0.30000000000000004, 'dense_units': 128, 'lr': 0.0004098521113058352, 'loss': 'bce'}


In [12]:
# --- BiLSTM tuner ----------------------------------------------------
def build_tunable_lstm(hp):
    lstm_units_1 = hp.Choice("lstm_units_1", [64, 128, 256])
    lstm_units_2 = hp.Choice("lstm_units_2", [32, 64, 128])
    dropout = hp.Float("dropout", 0.2, 0.5, step=0.1)
    dense_units = hp.Choice("dense_units", [64, 128, 256])
    lr = hp.Float("lr", 1e-4, 3e-3, sampling="log")
    loss_type = hp.Choice("loss", ["bce", "focal"])

    model = build_lstm(dropout=dropout, dense_units=dense_units,
                       lstm_units_1=lstm_units_1, lstm_units_2=lstm_units_2)
    loss = binary_focal_loss() if loss_type == "focal" else "binary_crossentropy"
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=loss,
        metrics=[keras.metrics.AUC(name="auc", multi_label=True)],
    )
    return model


if TUNE:
    tuner_lstm = kt.Hyperband(
        build_tunable_lstm,
        objective=kt.Objective("val_auc", direction="max"),
        max_epochs=TUNER_MAX_EPOCHS,
        factor=3,
        directory=TUNER_DIR,
        project_name="ecg_lstm",
        overwrite=False,
        seed=SEED,
    )
    tuner_lstm.search_space_summary()


Search space summary
Default search space size: 6
lstm_units_1 (Choice)
{'default': 64, 'conditions': [], 'values': [64, 128, 256], 'ordered': True}
lstm_units_2 (Choice)
{'default': 32, 'conditions': [], 'values': [32, 64, 128], 'ordered': True}
dropout (Float)
{'default': 0.2, 'conditions': [], 'min_value': 0.2, 'max_value': 0.5, 'step': 0.1, 'sampling': 'linear'}
dense_units (Choice)
{'default': 64, 'conditions': [], 'values': [64, 128, 256], 'ordered': True}
lr (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.003, 'step': None, 'sampling': 'log'}
loss (Choice)
{'default': 'bce', 'conditions': [], 'values': ['bce', 'focal'], 'ordered': False}


In [13]:
if TUNE:
    tuner_lstm.search(
        train_ds,
        validation_data=val_ds,
        epochs=TUNER_MAX_EPOCHS,
        callbacks=[callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                           patience=4, restore_best_weights=True)],
        verbose=1,
    )
    best_hps_lstm = tuner_lstm.get_best_hyperparameters(num_trials=1)[0]
    best_config_lstm = {k: best_hps_lstm.get(k) for k in
                        ["lstm_units_1", "lstm_units_2", "dropout",
                         "dense_units", "lr", "loss"]}
    print("Best BiLSTM hyperparameters:", best_config_lstm)

    with open(os.path.join(ART_DIR, "tuner_results_lstm.json"), "w") as f:
        json.dump(best_config_lstm, f, indent=2)
else:
    best_config_lstm = {"lstm_units_1": 128, "lstm_units_2": 64, "dropout": 0.3,
                        "dense_units": 128, "lr": 1e-3, "loss": "focal"}
    print("BiLSTM tuning skipped. Using defaults:", best_config_lstm)


Trial 30 Complete [00h 03m 36s]
val_auc: 0.9204921722412109

Best val_auc So Far: 0.924898624420166
Total elapsed time: 00h 41m 18s
Best BiLSTM hyperparameters: {'lstm_units_1': 128, 'lstm_units_2': 128, 'dropout': 0.2, 'dense_units': 128, 'lr': 0.00039447639218573435, 'loss': 'bce'}


## 8. Final training runs

In [14]:
# MLflow setup — optional, safe to skip if not installed
try:
    import mlflow
    mlflow.set_tracking_uri("file:./mlruns")
    mlflow.set_experiment("ECG_PTBXL_Final")
    USE_MLFLOW = True
except Exception as e:
    print(f"MLflow not available ({e}); continuing without tracking.")
    USE_MLFLOW = False


def train_model(model, name, loss_name="focal", lr=1e-3,
                epochs=50, use_augmentation=True):
    """Train a model and save to ecg_{name}_{aug|noaug}_final.keras.

    The _aug / _noaug suffix is chosen automatically from use_augmentation,
    so the augmented and non-augmented variants don't overwrite each other.
    """
    loss_fn = binary_focal_loss() if loss_name == "focal" else "binary_crossentropy"
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=loss_fn,
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc", multi_label=True),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

    suffix = "aug" if use_augmentation else "noaug"
    ckpt_path = os.path.join(ART_DIR, f"ecg_{name}_{suffix}_final.keras")
    cb = [
        callbacks.ModelCheckpoint(ckpt_path, monitor="val_auc", mode="max",
                                  save_best_only=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                    patience=5, min_lr=1e-6, verbose=1),
        callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                patience=10, restore_best_weights=True, verbose=1),
    ]

    tr_ds = train_ds if use_augmentation else make_dataset(
        X_train, y_train, BATCH_SIZE, augment=False, shuffle=True)

    run_name = f"{name}_{suffix}"
    if USE_MLFLOW:
        with mlflow.start_run(run_name=run_name):
            mlflow.log_params({"loss": loss_name, "lr": lr, "epochs": epochs,
                               "augmentation": use_augmentation,
                               "n_params": model.count_params()})
            history = model.fit(tr_ds, validation_data=val_ds,
                                epochs=epochs, callbacks=cb, verbose=2)
            for k, v in history.history.items():
                for i, val in enumerate(v):
                    mlflow.log_metric(k, val, step=i)
    else:
        history = model.fit(tr_ds, validation_data=val_ds,
                            epochs=epochs, callbacks=cb, verbose=2)
    return history


2026/04/18 16:01:15 INFO mlflow.tracking.fluent: Experiment with name 'ECG_PTBXL_Final' does not exist. Creating a new experiment.


In [15]:
# CNN with AUGMENTATION (tuned hyperparameters)
mult = best_config_cnn["filters_mult"]
tuned_filters = tuple(max(8, int(f * mult)) for f in (32, 64, 128, 256))

cnn_model_aug = build_cnn(
    filters=tuned_filters,
    dropout=best_config_cnn["dropout"],
    dense_units=best_config_cnn["dense_units"],
)
cnn_aug_history = train_model(cnn_model_aug, "cnn",
                              loss_name=best_config_cnn["loss"],
                              lr=best_config_cnn["lr"],
                              epochs=50,
                              use_augmentation=True)


Epoch 1/50

Epoch 1: val_auc improved from None to 0.89843, saving model to artifacts/ecg_cnn_aug_final.keras

Epoch 1: finished saving model to artifacts/ecg_cnn_aug_final.keras
264/264 - 12s - 47ms/step - accuracy: 0.8446 - auc: 0.8641 - loss: 0.3585 - precision: 0.7423 - recall: 0.6114 - val_accuracy: 0.8508 - val_auc: 0.8984 - val_loss: 0.3416 - val_precision: 0.7700 - val_recall: 0.6016 - learning_rate: 4.0985e-04
Epoch 2/50

Epoch 2: val_auc improved from 0.89843 to 0.90685, saving model to artifacts/ecg_cnn_aug_final.keras

Epoch 2: finished saving model to artifacts/ecg_cnn_aug_final.keras
264/264 - 3s - 10ms/step - accuracy: 0.8714 - auc: 0.9028 - loss: 0.3069 - precision: 0.7872 - recall: 0.6891 - val_accuracy: 0.8574 - val_auc: 0.9069 - val_loss: 0.3319 - val_precision: 0.7728 - val_recall: 0.6337 - learning_rate: 4.0985e-04
Epoch 3/50

Epoch 3: val_auc improved from 0.90685 to 0.90994, saving model to artifacts/ecg_cnn_aug_final.keras

Epoch 3: finished saving model to arti

In [16]:
# CNN without AUGMENTATION (same tuned hyperparameters for fair ablation)
cnn_model_noaug = build_cnn(
    filters=tuned_filters,
    dropout=best_config_cnn["dropout"],
    dense_units=best_config_cnn["dense_units"],
)
cnn_noaug_history = train_model(cnn_model_noaug, "cnn",
                                loss_name=best_config_cnn["loss"],
                                lr=best_config_cnn["lr"],
                                epochs=50,
                                use_augmentation=False)


Epoch 1/50

Epoch 1: val_auc improved from None to 0.89944, saving model to artifacts/ecg_cnn_noaug_final.keras

Epoch 1: finished saving model to artifacts/ecg_cnn_noaug_final.keras
264/264 - 12s - 47ms/step - accuracy: 0.8425 - auc: 0.8630 - loss: 0.3616 - precision: 0.7331 - recall: 0.6155 - val_accuracy: 0.8592 - val_auc: 0.8994 - val_loss: 0.3332 - val_precision: 0.7851 - val_recall: 0.6256 - learning_rate: 4.0985e-04
Epoch 2/50

Epoch 2: val_auc improved from 0.89944 to 0.90949, saving model to artifacts/ecg_cnn_noaug_final.keras

Epoch 2: finished saving model to artifacts/ecg_cnn_noaug_final.keras
264/264 - 3s - 11ms/step - accuracy: 0.8695 - auc: 0.9016 - loss: 0.3099 - precision: 0.7843 - recall: 0.6837 - val_accuracy: 0.8625 - val_auc: 0.9095 - val_loss: 0.3115 - val_precision: 0.7708 - val_recall: 0.6649 - learning_rate: 4.0985e-04
Epoch 3/50

Epoch 3: val_auc improved from 0.90949 to 0.91385, saving model to artifacts/ecg_cnn_noaug_final.keras

Epoch 3: finished saving mod

In [17]:
# BiLSTM with AUGMENTATION (tuned hyperparameters)
lstm_model_aug = build_lstm(
    dropout=best_config_lstm["dropout"],
    dense_units=best_config_lstm["dense_units"],
    lstm_units_1=best_config_lstm["lstm_units_1"],
    lstm_units_2=best_config_lstm["lstm_units_2"],
)
lstm_aug_history = train_model(lstm_model_aug, "lstm",
                               loss_name=best_config_lstm["loss"],
                               lr=best_config_lstm["lr"],
                               epochs=40,
                               use_augmentation=True)


Epoch 1/40

Epoch 1: val_auc improved from None to 0.88547, saving model to artifacts/ecg_lstm_aug_final.keras

Epoch 1: finished saving model to artifacts/ecg_lstm_aug_final.keras
264/264 - 16s - 60ms/step - accuracy: 0.8202 - auc: 0.8204 - loss: 0.4034 - precision: 0.7049 - recall: 0.5248 - val_accuracy: 0.8554 - val_auc: 0.8855 - val_loss: 0.3369 - val_precision: 0.7729 - val_recall: 0.6226 - learning_rate: 3.9448e-04
Epoch 2/40

Epoch 2: val_auc improved from 0.88547 to 0.89542, saving model to artifacts/ecg_lstm_aug_final.keras

Epoch 2: finished saving model to artifacts/ecg_lstm_aug_final.keras
264/264 - 12s - 45ms/step - accuracy: 0.8602 - auc: 0.8824 - loss: 0.3322 - precision: 0.7711 - recall: 0.6539 - val_accuracy: 0.8572 - val_auc: 0.8954 - val_loss: 0.3325 - val_precision: 0.7520 - val_recall: 0.6664 - learning_rate: 3.9448e-04
Epoch 3/40

Epoch 3: val_auc improved from 0.89542 to 0.90638, saving model to artifacts/ecg_lstm_aug_final.keras

Epoch 3: finished saving model t

In [18]:
# BiLSTM without AUGMENTATION (same tuned hyperparameters for fair ablation)
lstm_model_noaug = build_lstm(
    dropout=best_config_lstm["dropout"],
    dense_units=best_config_lstm["dense_units"],
    lstm_units_1=best_config_lstm["lstm_units_1"],
    lstm_units_2=best_config_lstm["lstm_units_2"],
)
lstm_noaug_history = train_model(lstm_model_noaug, "lstm",
                                 loss_name=best_config_lstm["loss"],
                                 lr=best_config_lstm["lr"],
                                 epochs=40,
                                 use_augmentation=False)


Epoch 1/40

Epoch 1: val_auc improved from None to 0.88276, saving model to artifacts/ecg_lstm_noaug_final.keras

Epoch 1: finished saving model to artifacts/ecg_lstm_noaug_final.keras
264/264 - 13s - 50ms/step - accuracy: 0.8233 - auc: 0.8271 - loss: 0.3985 - precision: 0.7081 - recall: 0.5391 - val_accuracy: 0.8509 - val_auc: 0.8828 - val_loss: 0.3469 - val_precision: 0.7543 - val_recall: 0.6263 - learning_rate: 3.9448e-04
Epoch 2/40

Epoch 2: val_auc improved from 0.88276 to 0.89544, saving model to artifacts/ecg_lstm_noaug_final.keras

Epoch 2: finished saving model to artifacts/ecg_lstm_noaug_final.keras
264/264 - 11s - 40ms/step - accuracy: 0.8582 - auc: 0.8829 - loss: 0.3333 - precision: 0.7686 - recall: 0.6468 - val_accuracy: 0.8614 - val_auc: 0.8954 - val_loss: 0.3365 - val_precision: 0.7724 - val_recall: 0.6561 - learning_rate: 3.9448e-04
Epoch 3/40

Epoch 3: val_auc improved from 0.89544 to 0.90087, saving model to artifacts/ecg_lstm_noaug_final.keras

Epoch 3: finished savi

## 9. Save histories for the comparison notebook

In [19]:
histories = {
    "cnn_aug":     cnn_aug_history.history,
    "cnn_noaug":   cnn_noaug_history.history,
    "lstm_aug":    lstm_aug_history.history,
    "lstm_noaug":  lstm_noaug_history.history,
}
with open(os.path.join(ART_DIR, "histories.pkl"), "wb") as f:
    pickle.dump(histories, f)

print("Training complete. Artifacts saved:")
for f in sorted(os.listdir(ART_DIR)):
    size_mb = os.path.getsize(os.path.join(ART_DIR, f)) / 1e6
    print(f"  {f:40s} {size_mb:>8.2f} MB")


Training complete. Artifacts saved:
  ecg_cnn_aug_final.keras                      8.43 MB
  ecg_cnn_noaug_final.keras                    8.43 MB
  ecg_lstm_aug_final.keras                    10.23 MB
  ecg_lstm_noaug_final.keras                  10.23 MB
  eval_sets.npz                              203.17 MB
  histories.pkl                                0.01 MB
  normalisation_params.npz                     0.00 MB
  ptbxl_signals_100hz.npy                   1012.94 MB
  tuner_results.json                           0.00 MB
  tuner_results_lstm.json                      0.00 MB


---

**Next step:** open `comparison.ipynb` for full evaluation and model comparison.